In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0 peft')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'common.py': '"""Shared helpers for GreenScan training on Kaggle (T4) and on a rented A100 (vast.ai).\n\nData is looked up in $GS_DATA, ./data, ../data, then /kaggle/input (a private dataset). Outputs go to $GS_OUT\n(default ./out). Every script prints "[cell] ..." at each stage and flushes, so a log shows where a run is.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef cell(name: str) -> None:\n    print(f"[cell] {name}", flush=True)\n\n\ndef log(*a) -> None:\n    print(*a, flush=True)\n\n\ndef find(name: str) -> Path:\n    for root in (os.environ.get("GS_DATA", ""), "data", "../data", "/kaggle/input"):\n        if root and Path(root).exists():\n            hit = next(iter(sorted(Path(root).rglob(name))), None)\n            if hit:\n                return hit\n    raise FileNotFoundError(name)\n\n\ndef has(name: str) -> bool:\n    try:\n        find(name)\n        return True\n    except FileNotFoundError:\n        return False\n\n\ndef jl(path) -> list[dict]:\n    p = Path(path) if not isinstance(path, str) or Path(path).exists() else find(path)\n    return [json.loads(x) for x in open(p, encoding="utf-8") if x.strip()]\n\n\ndef wjl(path, rows) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as fh:\n        for r in rows:\n            fh.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n\ndef out_dir(sub: str = "") -> Path:\n    p = Path(os.environ.get("GS_OUT", "out")) / sub\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef save_json(path, obj) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef norm60(s: str) -> str:\n    return re.sub(r"\\s+", " ", (s or "").lower()).strip()[:60]\n\n\ndef company_folds(groups: list[str], k: int = 5, seed: int = 13) -> list[tuple[list[int], list[int]]]:\n    """Out-of-company folds: every company sits in exactly one test fold; folds balanced by row count."""\n    count = Counter(groups)\n    comps = sorted(count, key=lambda c: (-count[c], c))\n    rng = random.Random(seed)\n    rng.shuffle(comps)\n    comps.sort(key=lambda c: -count[c])\n    load, fold_of = [0] * k, {}\n    for c in comps:\n        f = min(range(k), key=lambda i: load[i])\n        fold_of[c] = f\n        load[f] += count[c]\n    out = []\n    for f in range(k):\n        test = [i for i, g in enumerate(groups) if fold_of[g] == f]\n        train = [i for i, g in enumerate(groups) if fold_of[g] != f]\n        if test and train:\n            out.append((train, test))\n    return out\n\n\ndef prf(gold: list[str], pred: list[str], labels: list[str]) -> dict:\n    per = {}\n    for l in labels:\n        tp = sum(g == l and p == l for g, p in zip(gold, pred))\n        fp = sum(g != l and p == l for g, p in zip(gold, pred))\n        fn = sum(g == l and p != l for g, p in zip(gold, pred))\n        pr = tp / (tp + fp) if tp + fp else 0.0\n        rc = tp / (tp + fn) if tp + fn else 0.0\n        per[l] = {"precision": round(pr, 4), "recall": round(rc, 4),\n                  "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn, "predicted": tp + fp}\n    present = [l for l in labels if per[l]["support"]]\n    macro = sum(per[l]["f1"] for l in present) / len(present) if present else 0.0\n    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold) if gold else 0.0\n    return {"macro_f1": round(macro, 4), "accuracy": round(acc, 4), "per_class": per, "n": len(gold)}\n\n\ndef confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:\n    idx = {l: i for i, l in enumerate(labels)}\n    m = [[0] * len(labels) for _ in labels]\n    for g, p in zip(gold, pred):\n        if g in idx and p in idx:\n            m[idx[g]][idx[p]] += 1\n    return m\n\n\ndef boot_ci(items: list, groups: list[str], stat, n: int = 1000, seed: int = 7) -> list[float] | None:\n    """95% interval of stat(items) resampling whole companies (cluster bootstrap)."""\n    by = defaultdict(list)\n    for it, g in zip(items, groups):\n        by[g].append(it)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(n):\n        sample = [it for _ in keys for it in by[rng.choice(keys)]]\n        try:\n            vals.append(stat(sample))\n        except ZeroDivisionError:\n            continue\n    vals.sort()\n    return [round(vals[int(0.025 * len(vals))], 4), round(vals[int(0.975 * len(vals)) - 1], 4)] if vals else None\n\n\nclass Timer:\n    def __init__(self):\n        self.t0 = time.time()\n\n    def __call__(self) -> float:\n        return round(time.time() - self.t0, 1)\n',
    'train_stance_enc.py': '"""Offline stance model (CPU): mDeBERTa-v3-base-xnli fine-tuned as a 4-relation cross-encoder (claim, passage).\n\n    python -u train_stance_enc.py [--epochs 4] [--folds 5]\n\nLabels: pairs_labels.jsonl (Claude A/B + adjudication, CAP v1.0). Pairs whose claim overlaps GS-Eval are dropped so S5\nstays an external test. Loss: class-weighted cross-entropy (∝ 1/sqrt(frequency)); temperature fitted on out-of-fold\nlogits. A relation the labels never contain (CONTRADICTS is almost absent) is never predicted alone: the model abstains\nto CONTEXT unless P(CONTRADICTS) >= 0.9.\nReported (out/stance/metrics.json): out-of-company CV on the pairs (macro-F1, per class, confusion, risk-coverage),\nbaselines on the same pairs (rules, zero-shot NLI), S5 external test (macro-F1 over S and O, PARTIAL counted as S,\nfalse C per 100) against the rules, CPU ms per pair. Model of the final fit: out/stance/mdeberta-stance-final/.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import (Timer, boot_ci, cell, company_folds, confusion, has, jl, log, out_dir, prf,  # noqa: E402\n                    save_json, seed_all)\n\nBASE = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"\nLABELS = ["SUPPORTS", "PARTIAL", "CONTRADICTS", "CONTEXT"]\nL2I = {l: i for i, l in enumerate(LABELS)}\nMAX_LEN, BATCH = 384, 16\nTO_SCO = {"SUPPORTS": "S", "PARTIAL": "S", "CONTRADICTS": "C", "CONTEXT": "O"}\n\n\ndef encode(tok, claims, passages, dev):\n    return tok(claims, passages, truncation="only_second", max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)\n\n\ndef fit(rows, epochs, lr, seed, hist, tag):\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n    seed_all(seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    tok = AutoTokenizer.from_pretrained(BASE)\n    model = AutoModelForSequenceClassification.from_pretrained(\n        BASE, num_labels=4, id2label=dict(enumerate(LABELS)), label2id=L2I, ignore_mismatched_sizes=True).float().to(dev)\n    cnt = Counter(L2I[r["relation"]] for r in rows)\n    w = torch.tensor([1 / math.sqrt(cnt[i]) if cnt.get(i) else 0.0 for i in range(4)], device=dev)\n    lossf = torch.nn.CrossEntropyLoss(weight=w * (sum(1 for x in w if x > 0)) / w.sum())\n    steps = epochs * math.ceil(len(rows) / BATCH)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    scaler = torch.amp.GradScaler(enabled=dev == "cuda")\n    model.train()\n    for ep in range(epochs):\n        order = list(range(len(rows)))\n        random.shuffle(order)\n        tot = 0.0\n        for i in range(0, len(order), BATCH):\n            b = [rows[j] for j in order[i:i + BATCH]]\n            x = encode(tok, [r["claim"] for r in b], [r["passage"] for r in b], dev)\n            y = torch.tensor([L2I[r["relation"]] for r in b], device=dev)\n            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):\n                loss = lossf(model(**x).logits.float(), y)\n            opt.zero_grad()\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt)\n            scaler.update()\n            sch.step()\n            tot += loss.item() * len(b)\n        hist.append({"run": tag, "epoch": ep + 1, "loss": round(tot / len(rows), 4)})\n        log(f"[stance] {tag} epoch {ep + 1} loss {tot / len(rows):.4f}")\n    return model.eval(), tok\n\n\ndef logits(model, tok, claims, passages):\n    import torch\n    dev = next(model.parameters()).device\n    out = []\n    with torch.no_grad():\n        for i in range(0, len(claims), 32):\n            x = encode(tok, claims[i:i + 32], passages[i:i + 32], dev)\n            out += model(**x).logits.float().cpu().tolist()\n    return out\n\n\ndef softmax(z, t=1.0):\n    m = max(z)\n    e = [math.exp((v - m) / t) for v in z]\n    s = sum(e)\n    return [v / s for v in e]\n\n\ndef decide(p, c_min=0.9):\n    i = max(range(4), key=lambda k: p[k])\n    if LABELS[i] == "CONTRADICTS" and p[i] < c_min:\n        return "CONTEXT"\n    return LABELS[i]\n\n\ndef fit_temperature(z, y):\n    best = (1e9, 1.0)\n    for t in [0.5 + 0.1 * k for k in range(46)]:\n        nll = -sum(math.log(max(softmax(zi, t)[yi], 1e-12)) for zi, yi in zip(z, y)) / len(y)\n        best = min(best, (nll, t))\n    return round(best[1], 2), round(best[0], 4)\n\n\ndef risk_coverage(p, gold):\n    """Accuracy of the most confident fraction, for coverage 10% .. 100%."""\n    order = sorted(range(len(p)), key=lambda i: -max(p[i]))\n    out = []\n    for c in range(1, 11):\n        k = max(1, round(len(order) * c / 10))\n        sel = order[:k]\n        out.append({"coverage": c / 10, "accuracy": round(sum(decide(p[i]) == gold[i] for i in sel) / k, 4),\n                    "min_conf": round(max(p[sel[-1]]), 4)})\n    return out\n\n\ndef sco_scores(gold, pred, groups):\n    def mf(items):\n        g, p = [a for a, _ in items], [b for _, b in items]\n        r = prf(g, p, ["S", "O"])\n        return r["macro_f1"]\n    items = list(zip(gold, pred))\n    return {"macro_f1_SO": round(mf(items), 4), "ci95": boot_ci(items, groups, mf),\n            "false_C": sum(p == "C" and g != "C" for g, p in items), "C_found": sum(p == "C" and g == "C" for g, p in items),\n            "C_gold": sum(g == "C" for g in gold), "false_C_per_100": round(100 * sum(p == "C" and g != "C" for g, p in items) / len(items), 2),\n            "n": len(items), "report": prf(gold, pred, ["S", "C", "O"])}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--epochs", type=int, default=4)\n    ap.add_argument("--lr", type=float, default=2e-5)\n    ap.add_argument("--folds", type=int, default=5)\n    args = ap.parse_args()\n    import torch\n\n    cell("stance data")\n    pool = {r["id"]: r for r in jl("pairs_pool.jsonl")}\n    labels = jl("pairs_labels.jsonl")\n    rows = [{**pool[x["id"]], "relation": x["relation"]} for x in labels\n            if x["id"] in pool and not pool[x["id"]]["overlaps_gs_eval"] and x.get("decided_by") != "contested"]\n    groups = [r["company_id"] for r in rows]\n    log(f"[stance] {len(rows)} labelled pairs · {dict(Counter(r[\'relation\'] for r in rows))} · companies {len(set(groups))}")\n    od = out_dir("stance")\n    hist, m = [], {"n": len(rows), "relations": dict(Counter(r["relation"] for r in rows))}\n\n    cell("stance cv")\n    t = Timer()\n    oof = [None] * len(rows)\n    for f, (tr, te) in enumerate(company_folds(groups, args.folds)):\n        model, tok = fit([rows[i] for i in tr], args.epochs, args.lr, 13, hist, f"fold{f}")\n        for i, z in zip(te, logits(model, tok, [rows[i]["claim"] for i in te], [rows[i]["passage"] for i in te])):\n            oof[i] = z\n        del model\n        torch.cuda.empty_cache()\n    gold = [r["relation"] for r in rows]\n    temp, nll = fit_temperature(oof, [L2I[g] for g in gold])\n    probs = [softmax(z, temp) for z in oof]\n    pred = [decide(p) for p in probs]\n    m["cv"] = {**prf(gold, pred, LABELS), "confusion": confusion(gold, pred, LABELS), "labels": LABELS,\n               "temperature": temp, "nll": nll, "risk_coverage": risk_coverage(probs, gold), "seconds": t(),\n               "ci95_macro_f1": boot_ci(list(zip(gold, pred)), groups,\n                                        lambda it: prf([a for a, _ in it], [b for _, b in it], LABELS)["macro_f1"])}\n    base = {}\n    for name in ("rules", "nli-mdeberta"):\n        if has(f"pairs_cand_{name}.jsonl"):\n            cand = {c["id"]: c["relation"] for c in jl(f"pairs_cand_{name}.jsonl")}\n            bp = [cand.get(r["id"], "CONTEXT") for r in rows]\n            base[name] = {**prf(gold, bp, LABELS), "confusion": confusion(gold, bp, LABELS)}\n    m["cv_baselines"] = base\n    log(f"[stance] CV macro-F1 {m[\'cv\'][\'macro_f1\']} (T={temp}) · baselines "\n        f"{ {k: v[\'macro_f1\'] for k, v in base.items()} }")\n\n    cell("stance final + S5")\n    model, tok = fit(rows, args.epochs, args.lr, 13, hist, "final")\n    model.save_pretrained(od / "mdeberta-stance-final")\n    tok.save_pretrained(od / "mdeberta-stance-final")\n    s5 = jl("s5_pairs.jsonl")\n    z = logits(model, tok, [r["claim"] for r in s5], [r["evidence"] for r in s5])\n    pm = [TO_SCO[decide(softmax(v, temp))] for v in z]\n    pr = [TO_SCO.get(r.get("rules") or "CONTEXT", "O") for r in s5]\n    g5 = [r["gold"] for r in s5]\n    c5 = [r["company"] for r in s5]\n    m["s5"] = {"model": sco_scores(g5, pm, c5), "rules": sco_scores(g5, pr, c5)}\n    log(f"[stance] S5 model {m[\'s5\'][\'model\'][\'macro_f1_SO\']} {m[\'s5\'][\'model\'][\'ci95\']} false C "\n        f"{m[\'s5\'][\'model\'][\'false_C\']} · rules {m[\'s5\'][\'rules\'][\'macro_f1_SO\']} false C {m[\'s5\'][\'rules\'][\'false_C\']}")\n\n    cell("stance cpu latency")\n    cpu = model.to("cpu")\n    torch.set_num_threads(4)\n    t0 = time.time()\n    logits(cpu, tok, [r["claim"] for r in s5[:48]], [r["evidence"] for r in s5[:48]])\n    m["cpu_ms_per_pair_4threads"] = round((time.time() - t0) / 48 * 1000, 1)\n    save_json(od / "metrics.json", m)\n    save_json(od / "history.json", hist)\n    log(f"[stance] cpu {m[\'cpu_ms_per_pair_4threads\']} ms/pair · DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'train_llm_sft.py': '"""LoRA supervised fine-tuning of a chat model: loss = token cross-entropy on the answer only (prompt masked).\n\n    python -u train_llm_sft.py --base Qwen/Qwen3-4B-Instruct-2507 --train out/teacher/explain_teacher.jsonl \\\n        --name explain-4b --epochs 2 --dtype fp16 [--merge]          # Kaggle T4\n    python -u train_llm_sft.py --base Qwen/Qwen3-14B --train data/stance_sft.jsonl --name stance-14b --dtype bf16  # A100\n\nInput rows: {"id", "company", "messages": [system, user], "target": "<assistant answer>"}. About 10% of companies are\nheld out for the validation loss (out-of-company). Writes out/<name>/adapter (+ merged/ with --merge) and\nout/<name>/history.json (train loss every 10 steps, validation loss every epoch) for the charts.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, jl, log, out_dir, save_json, seed_all  # noqa: E402\n\n\ndef encode(tok, row, max_len):\n    prompt = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    p = tok(prompt, add_special_tokens=False)["input_ids"]\n    a = tok(row["target"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]\n    if len(p) + len(a) > max_len:\n        return None\n    return {"input_ids": p + a, "labels": [-100] * len(p) + a}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--train", required=True)\n    ap.add_argument("--name", required=True)\n    ap.add_argument("--epochs", type=float, default=2)\n    ap.add_argument("--lr", type=float, default=2e-4)\n    ap.add_argument("--r", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=2048)\n    ap.add_argument("--accum", type=int, default=8)\n    ap.add_argument("--dtype", choices=["fp16", "bf16"], default="fp16")\n    ap.add_argument("--val-frac", type=float, default=0.1)\n    ap.add_argument("--merge", action="store_true")\n    ap.add_argument("--merge-dir", default="", help="where the merged fp16 model goes (Kaggle: /tmp, outside the 20 GB output)")\n    ap.add_argument("--seed", type=int, default=13)\n    args = ap.parse_args()\n    seed_all(args.seed)\n\n    cell(f"sft load {args.base}")\n    import torch\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup\n    dtype = torch.float16 if args.dtype == "fp16" else torch.bfloat16\n    tok = AutoTokenizer.from_pretrained(args.base)\n    rows = jl(args.train)\n    comps = sorted({r.get("company") or "?" for r in rows})\n    random.Random(args.seed).shuffle(comps)\n    val_comps = set(comps[: max(1, round(len(comps) * args.val_frac))]) if len(comps) > 3 else set()\n    enc = [(r, encode(tok, r, args.max_len)) for r in rows]\n    dropped = sum(e is None for _, e in enc)\n    train = [e for r, e in enc if e and (r.get("company") or "?") not in val_comps]\n    val = [e for r, e in enc if e and (r.get("company") or "?") in val_comps]\n    log(f"[sft] rows {len(rows)} · train {len(train)} · val {len(val)} (companies {sorted(val_comps)}) · "\n        f"too long {dropped}")\n\n    model = AutoModelForCausalLM.from_pretrained(args.base, torch_dtype=dtype, device_map={"": 0})\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    model = get_peft_model(model, LoraConfig(r=args.r, lora_alpha=args.alpha, lora_dropout=0.05, task_type="CAUSAL_LM",\n                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj",\n                                                             "gate_proj", "up_proj", "down_proj"]))\n    model.print_trainable_parameters()\n    params = [p for p in model.parameters() if p.requires_grad]\n    steps = math.ceil(len(train) * args.epochs / args.accum)\n    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0)\n    sch = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * steps)), steps)\n    scaler = torch.amp.GradScaler(enabled=args.dtype == "fp16")\n    hist = {"train": [], "val": [], "config": vars(args), "train_rows": len(train), "val_rows": len(val)}\n\n    def loss_of(e):\n        ids = torch.tensor([e["input_ids"]], device="cuda")\n        lab = torch.tensor([e["labels"]], device="cuda")\n        with torch.autocast("cuda", dtype=dtype):\n            return model(input_ids=ids, labels=lab).loss\n\n    def evaluate():\n        if not val:\n            return None\n        model.eval()\n        with torch.no_grad():\n            v = [loss_of(e).float().item() for e in val]\n        model.train()\n        return round(sum(v) / len(v), 4)\n\n    cell("sft train")\n    t, step, seen, run, bad = Timer(), 0, 0, [], 0\n    hist["val"].append({"epoch": 0, "loss": evaluate()})\n    model.train()\n    total = int(len(train) * args.epochs)\n    order = []\n    while len(order) < total:\n        ep = list(range(len(train)))\n        random.shuffle(ep)\n        order += ep\n    for i, j in enumerate(order[:total], 1):\n        loss = loss_of(train[j]) / args.accum\n        if not torch.isfinite(loss):\n            bad += 1\n            if bad >= 5:\n                raise SystemExit("[sft] loss is not finite 5 times: try --dtype bf16 (A100) or a smaller --lr")\n            opt.zero_grad(set_to_none=True)\n            continue\n        scaler.scale(loss).backward()\n        run.append(loss.item() * args.accum)\n        if i % args.accum == 0 or i == total:\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0)\n            scaler.step(opt)\n            scaler.update()\n            opt.zero_grad(set_to_none=True)\n            sch.step()\n            step += 1\n            if step % 10 == 0 or i == total:\n                hist["train"].append({"step": step, "epoch": round(i / len(train), 3), "loss": round(sum(run) / len(run), 4),\n                                      "lr": sch.get_last_lr()[0]})\n                log(f"[sft] step {step}/{steps} epoch {i / len(train):.2f} loss {sum(run) / len(run):.4f} · {t()} s")\n                run = []\n        if i % len(train) == 0 and i < total:\n            v = evaluate()\n            hist["val"].append({"epoch": round(i / len(train), 2), "loss": v})\n            log(f"[sft] epoch {i / len(train):.0f} val loss {v}")\n    hist["val"].append({"epoch": args.epochs, "loss": evaluate()})\n    hist["seconds"] = t()\n    log(f"[sft] val losses {hist[\'val\']}")\n\n    cell("sft save")\n    od = out_dir(args.name)\n    model.save_pretrained(od / "adapter")\n    tok.save_pretrained(od / "adapter")\n    save_json(od / "history.json", hist)\n    if args.merge:\n        mdir = Path(args.merge_dir) if args.merge_dir else od / "merged"\n        merged = model.merge_and_unload()\n        merged.save_pretrained(mdir, safe_serialization=True)\n        tok.save_pretrained(mdir)\n        log(f"[sft] merged model -> {mdir}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'eval_stance_llm.py': '"""Score a chat model (and optionally its stance LoRA) on the registered S5 set: 391 GS-Eval pairs, gold S / C / O.\n\n    python -u eval_stance_llm.py --base Qwen/Qwen3-14B --adapter out/stance-14b/adapter --field msgs_large --tag q3-14b\n    python -u eval_stance_llm.py --base openai/gpt-oss-120b --field msgs_large --tag gpt-oss-120b\n\nAnswer {"relation": SUPPORTS|PARTIAL|CONTRADICTS|CONTEXT, "reason"}; PARTIAL counts as S (as train_stance_enc.py).\nAn unparsed answer counts as O (the product falls back to the rules). Metrics as llm_stages_v1: macro-F1 over S and O\nwith a company-cluster interval, false C per 100, C found. Writes out/stance_eval/<tag>.json.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, has, jl, log, out_dir, save_json, wjl  # noqa: E402\nfrom train_stance_enc import TO_SCO, sco_scores  # noqa: E402\n\n\ndef relation(text: str) -> str | None:\n    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)\n    m = re.search(r"\\{.*?\\}", text, flags=re.S)\n    if m:\n        try:\n            r = str(json.loads(m.group(0)).get("relation", "")).upper()\n            if r in TO_SCO:\n                return r\n        except json.JSONDecodeError:\n            pass\n    m = re.search(r"SUPPORTS|PARTIAL|CONTRADICTS|CONTEXT", text or "")\n    return m.group(0) if m else None\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", default="")\n    ap.add_argument("--tp", type=int, default=1)\n    ap.add_argument("--field", default="msgs_large")\n    ap.add_argument("--tag", default="")\n    ap.add_argument("--max-len", type=int, default=6144)\n    ap.add_argument("--data", default="s5_pairs.jsonl", help="or stance_sft.jsonl: the 821 silver pairs (gold = target)")\n    args = ap.parse_args()\n    cell("stance llm load")\n    data = jl(args.data)\n    if data and "target" in data[0]:   # silver pairs: gold relation from the SFT target, rules from the candidates file\n        rules = {c["id"]: c["relation"] for c in jl("pairs_cand_rules.jsonl")} if has("pairs_cand_rules.jsonl") else {}\n        for d in data:\n            d["gold"] = TO_SCO[json.loads(d["target"])["relation"]]\n            d["rules"] = rules.get(d["id"])\n            d.setdefault(args.field, d["messages"])\n    from transformers import AutoTokenizer\n    from vllm import LLM, SamplingParams\n    tok = AutoTokenizer.from_pretrained(args.base)\n    prompts = [tok.apply_chat_template(d[args.field], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n               for d in data]\n    kw = dict(enable_lora=True, max_lora_rank=64) if args.adapter else {}\n    llm = LLM(model=args.base, tensor_parallel_size=args.tp, max_model_len=args.max_len, gpu_memory_utilization=0.9,\n              seed=0, **kw)\n    systems = [("base", None)]\n    if args.adapter:\n        from vllm.lora.request import LoRARequest\n        systems.append(("lora", LoRARequest("stance", 1, args.adapter)))\n    tag = args.tag or Path(args.base).name\n    od = out_dir("stance_eval")\n    gold, comp = [d["gold"] for d in data], [d["company"] for d in data]\n    res = {"base": args.base, "adapter": args.adapter, "data": args.data, "systems": {},\n           "rules": sco_scores(gold, [TO_SCO.get(d.get("rules") or "CONTEXT", "O") for d in data], comp)}\n    for name, lora in systems:\n        cell(f"stance llm generate {name}")\n        t = Timer()\n        outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=200), lora_request=lora)\n        rel = [relation(o.outputs[0].text) for o in outs]\n        pred = [TO_SCO[r] if r else "O" for r in rel]\n        s = sco_scores(gold, pred, comp)\n        s.update(seconds=t(), unparsed=sum(r is None for r in rel))\n        res["systems"][name] = s\n        wjl(od / f"{tag}_{name}_answers.jsonl", [{"id": d["id"], "text": o.outputs[0].text, "relation": r}\n                                                  for d, o, r in zip(data, outs, rel)])\n        log(f"[stance-llm] {tag} {name}: macro-F1(S,O) {s[\'macro_f1_SO\']} {s[\'ci95\']} false C {s[\'false_C\']} "\n            f"unparsed {s[\'unparsed\']}")\n    save_json(od / f"{tag}.json", res)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u train_llm_sft.py --base Qwen/Qwen3-4B-Instruct-250', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_llm_sft.py --base Qwen/Qwen3-4B-Instruct-2507 --train stance_sft.jsonl --name stance-4b --epochs 3 --lr 1e-4 --dtype fp16')


In [ ]:
print('[cell] run 2: python -u eval_stance_llm.py --base Qwen/Qwen3-4B-Instruct-2', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u eval_stance_llm.py --base Qwen/Qwen3-4B-Instruct-2507 --adapter out/stance-4b/adapter --field msgs_large --tag S5-q3-4b-lora')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)